# Company Sales Brochure Generator

Create a product which builds a **sales brochure** for a company to be used for prospective clients, investors and potential recruits.

**Input:** company name and their main website

**Tools & Techniques:**
- OpenAI API (Chat Completions API)
- One-shot prompting
- Streaming results as markdown, json
- Convert this into a commercial solution

It will involve pulling multiple web pages: by starting with the primary webpage, scanning it for relavant links, and then pulling the content of those links.

Will include compounding LLM calls (multiple LLM calls) to get better output.

**GOAL:** Work as an AI Engineer, and build a solution using GPT models.

In [1]:
# imports
import os
import json
from IPython.display import display, Markdown
from openai import OpenAI
from dotenv import load_dotenv

In [2]:
# load openai api key
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_TUTORIALS_API_KEY")

if not "sk-proj" in OPENAI_API_KEY:
  raise ValueError("Please set your OpenAI API key in the .env file")
else:
  print("OpenAI API key loaded successfully")

OpenAI API key loaded successfully


In [3]:
llm = OpenAI(api_key=OPENAI_API_KEY)

In [4]:
# function to scrape a website and get all its links (anchor tags)
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def fetch_website_links(url):
    """
    Scrape a website and return a list of all links found in the webpage.
    
    Args:
        url (str): The URL of the website to scrape.
    
    Returns:
        list: A list of absolute URLs found in the webpage.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        links = []
        for a_tag in soup.find_all('a', href=True):
            link = urljoin(url, a_tag['href'])
            links.append(link)
        return links
    except requests.RequestException as e:
        print(f"Error fetching the website: {e}")
        return []
    except Exception as e:
        print(f"An error occurred: {e}")
        return []

In [5]:
# testing out this function
test_links = fetch_website_links("https://www.sarvam.ai/")
# https://simplysouthindia.com/
unique_links = set(test_links)
print(len(unique_links))
for link in unique_links:
    print(link)

22
https://www.sarvam.ai/apis/document-digitisation
https://www.sarvam.ai/
https://www.sarvam.ai/products/conversational-agents
https://www.sarvam.ai/privacy-policy
https://dashboard.sarvam.ai/
https://www.sarvam.ai/blogs/evaluating-indian-language-asr
https://www.sarvam.ai/blogs
https://www.sarvam.ai/about-us
https://www.sarvam.ai/products/akshar
https://www.sarvam.ai/terms-of-use
https://www.sarvam.ai/stories/tata-capital-ai-voice-transformation
https://www.linkedin.com/company/sarvam-ai
https://www.sarvam.ai/apis/speech-to-text
https://x.com/sarvamai
https://www.sarvam.ai/products/studio
https://discord.com/invite/5rAsykttcs
https://www.sarvam.ai/careers
https://youtube.com/@sarvamai
https://www.sarvam.ai/api-pricing
https://www.sarvam.ai/blogs/introducing-indus
https://www.sarvam.ai/blogs/sarvam-30b-105b
https://www.sarvam.ai/apis/text-to-speech


## Figure out the relavant links using an LLM

Invoke an LLM to decide what links are relavant. Doing this manually for a lot of links is hard.

In the below code, we will let LLM **output data in a structured format**. Additionally, we will use **one-shot prompting** technique, where we provide one example of what the output should look like.

Think of the following:
1. System Prompt (Instructions to the LLM)
2. User Prompt (Input to the LLM)

**Why output in JSON?**
- LLMs are trained on lots of data in one of these formats:
  - Plain english
  - Markdown
  - JSON
- Using structured input / output format, LLMs behave more coherently. Its a good way to express information and expect good results.

**Giving examples in the prompt:**
- One shot prompting: give one example of how the output should look like.
- Multi shot prompting: give multiple examples of how the output should look like.
- Zero shot prompting: don't give any examples of how the output should look like.

In [6]:
system_prompt = """
You are an expert AI agent who can look at a given set of links, and decide which ones are relevant 
to be included in a brochure to a company where the target audience could be prospective clients, investors, recruits, etc.

The useful links could be like the about page, careers page, products page, etc.

The links should be returned in a JSON format as follows:
{
  "links": [
    {"name": "About Us", "url": "https://example.com/about"},
    {"name": "Careers", "url": "https://example.com/careers"},
    {"name": "Products", "url": "https://example.com/products"}
  ]
}
"""

In [7]:
# construct the user prompt, which contains the list of links as well
def generate_user_prompt(website_name, url):
  links = fetch_website_links(url)
  user_prompt = f"""
Below are the list of links found in the website of {website_name} at the URL {url}.
Your task is to decide what links are relavant to create a brochure for the company, where the potential stake holders are prospective clients, investors, recruits, etc.
Do not include links about non-relavant items like Terms of Service, Privacy Policy, emails, links to youtube, etc. But links to other social media like Twitter, LinkedIn, Instagram might be relavant ones.

Links:
{"\n".join(links)}
  """
  return user_prompt

In [8]:
print(generate_user_prompt("Sarvam.AI", "https://www.sarvam.ai/"))


Below are the list of links found in the website of Sarvam.AI at the URL https://www.sarvam.ai/.
Your task is to decide what links are relavant to create a brochure for the company, where the potential stake holders are prospective clients, investors, recruits, etc.
Do not include links about non-relavant items like Terms of Service, Privacy Policy, emails, links to youtube, etc. But links to other social media like Twitter, LinkedIn, Instagram might be relavant ones.

Links:
https://www.sarvam.ai/
https://www.sarvam.ai/
https://dashboard.sarvam.ai/
https://www.sarvam.ai/stories/tata-capital-ai-voice-transformation
https://www.sarvam.ai/blogs/evaluating-indian-language-asr
https://www.sarvam.ai/blogs/sarvam-30b-105b
https://www.sarvam.ai/blogs/introducing-indus
https://www.sarvam.ai/blogs
https://www.sarvam.ai/
https://www.sarvam.ai/products/conversational-agents
https://www.sarvam.ai/products/studio
https://www.sarvam.ai/products/akshar
https://www.sarvam.ai/apis/text-to-speech
https

In [9]:
# function to invoke the LLM to filter out the relevant links

def filter_relavant_links_for_brochure(website_name, url):
  response = llm.chat.completions.create(
    model="gpt-5-nano",
    messages=[
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": generate_user_prompt(website_name, url)}
    ],
    response_format={"type": "json_object"} # return the response as a JSON object
  )
  print("Response:")
  print(response, end="\n-----\n")
  json_string = response.choices[0].message.content
  print("JSON string:")
  print(json_string, end="\n-----\n")
  json_response = json.loads(json_string)
  return json_response

`response_format={"type": "json_object"}` will force the model to predict the tokens such that the overall output will be a valid JSON.

In [10]:
relavant_links = filter_relavant_links_for_brochure("Sarvam.AI", "https://www.sarvam.ai/")

Response:
ChatCompletion(id='chatcmpl-DTibpzp1nQvC7flezqctdrXarqSTK', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "links": [\n    {"name": "Sarvam.AI Home", "url": "https://www.sarvam.ai/"},\n    {"name": "About Us", "url": "https://www.sarvam.ai/about-us"},\n    {"name": "Careers", "url": "https://www.sarvam.ai/careers"},\n    {"name": "API Pricing", "url": "https://www.sarvam.ai/api-pricing"},\n    {"name": "Conversational Agents", "url": "https://www.sarvam.ai/products/conversational-agents"},\n    {"name": "Studio", "url": "https://www.sarvam.ai/products/studio"},\n    {"name": "Akshar", "url": "https://www.sarvam.ai/products/akshar"},\n    {"name": "Text to Speech API", "url": "https://www.sarvam.ai/apis/text-to-speech"},\n    {"name": "Speech to Text API", "url": "https://www.sarvam.ai/apis/speech-to-text"},\n    {"name": "Document Digitisation API", "url": "https://www.sarvam.ai/apis/document-digitisation"},\n    {"na

In [11]:
langchain_relavant_links = filter_relavant_links_for_brochure("Langchain", "https://www.langchain.com/")

Response:
ChatCompletion(id='chatcmpl-DTicKcDk9UKq4oFqvDcoOHQsuCVXx', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "links": [\n    {"name": "LangChain - Home", "url": "https://www.langchain.com/"},\n    {"name": "About LangChain", "url": "https://www.langchain.com/about"},\n    {"name": "Careers", "url": "https://www.langchain.com/careers"},\n    {"name": "LangChain Partner Network", "url": "https://www.langchain.com/langchain-partner-network"},\n    {"name": "Customers", "url": "https://www.langchain.com/customers"},\n    {"name": "Resources", "url": "https://www.langchain.com/resources"},\n    {"name": "Blog", "url": "https://blog.langchain.com"},\n    {"name": "Documentation (Docs)", "url": "https://docs.langchain.com/"},\n    {"name": "Academy", "url": "https://academy.langchain.com/"},\n    {"name": "Startups", "url": "https://www.langchain.com/startups"},\n    {"name": "Pricing", "url": "https://www.langchain.com/pricin

## Scrape content from all the relavant links

Scrape all relavant links, get their content, and glue together all the content.

This is the input to the LLM when generate the brochure.

In [12]:
# function to scrape content of a given link
def fetch_website_content(url):
    """
    Scrape a website and return its content as text.
    
    Args:
        url (str): The URL of the website to scrape.
    
    Returns:
        str: The text content of the website.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        # Remove script and style elements
        for script in soup(["script", "style"]):
            script.extract()
        # Get text
        text = soup.get_text()
        # Clean up whitespace
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        text = '\n'.join(chunk for chunk in chunks if chunk)
        return text
    except requests.RequestException as e:
        print(f"Error fetching the website: {e}")
        return ""
    except Exception as e:
        print(f"An error occurred: {e}")
        return ""

In [13]:
# test this out

content = fetch_website_content(relavant_links['links'][0]['url'])
print(content)

Sarvam | India's Full-Stack Sovereign AI Platform
PlatformDevelopersResourcesCompanyExperience SarvamTalk to Sales
India's Sovereign AI PlatformAI for all from IndiaBuilt on sovereign compute. Powered by frontier-class models.Delivering population-scale impact.Experience Sarvam
India builds with Sarvam
Powering India's AI-first futureSovereign by designBuild, deploy, and run AI with full control, developed and operated entirely in IndiaState of the art modelsIndustry-leading models built for India's languages, culture, and contextHuman at the coreForward deployed engineers work alongside your teams to deliver production-ready agents
For Enterprise | Government | DevelopersIndia's full-stack sovereign AI platformPopulation-scale ApplicationsBuilding products India can use. Conversational agents fluent in India's languages. Platforms that run enterprise workflows from start to finish.SamvaadStudioState-of-the-art ModelsState-of-the-art models trained on sovereign data, delivering strong 

In [14]:
# pretty print relavant links JSON for visualization
print(json.dumps(relavant_links, indent=2))

{
  "links": [
    {
      "name": "Sarvam.AI Home",
      "url": "https://www.sarvam.ai/"
    },
    {
      "name": "About Us",
      "url": "https://www.sarvam.ai/about-us"
    },
    {
      "name": "Careers",
      "url": "https://www.sarvam.ai/careers"
    },
    {
      "name": "API Pricing",
      "url": "https://www.sarvam.ai/api-pricing"
    },
    {
      "name": "Conversational Agents",
      "url": "https://www.sarvam.ai/products/conversational-agents"
    },
    {
      "name": "Studio",
      "url": "https://www.sarvam.ai/products/studio"
    },
    {
      "name": "Akshar",
      "url": "https://www.sarvam.ai/products/akshar"
    },
    {
      "name": "Text to Speech API",
      "url": "https://www.sarvam.ai/apis/text-to-speech"
    },
    {
      "name": "Speech to Text API",
      "url": "https://www.sarvam.ai/apis/speech-to-text"
    },
    {
      "name": "Document Digitisation API",
      "url": "https://www.sarvam.ai/apis/document-digitisation"
    },
    {
     

In [15]:
def generate_website_content_context(website_name: str, links: list) -> str:
  """
  Input: website_name - string
  Input: links - list of dictionaries with fields: name, url
  """
  content = """
  Website Name: {website_name}

  Relavant Links & their Content:

  """
  for link in links:
    link_content = fetch_website_content(link['url'])
    content += f"""
    --------------

    Link Name: {link['name']}
    Link URL: {link['url']}
    Link Content: 
    {link_content}

    """
  return content

In [16]:
company_context = generate_website_content_context("Sarvam.AI", filter_relavant_links_for_brochure("Sarvam.AI", "https://www.sarvam.ai/")['links'])

print(company_context)

Response:
ChatCompletion(id='chatcmpl-DTidYFbrZP14b182sCaaCxJzJ6gvx', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "links": [\n    {"name": "Home", "url": "https://www.sarvam.ai/"},\n    {"name": "About Us", "url": "https://www.sarvam.ai/about-us"},\n    {"name": "Careers", "url": "https://www.sarvam.ai/careers"},\n    {"name": "Tata Capital AI Voice Transformation (Case Study)", "url": "https://www.sarvam.ai/stories/tata-capital-ai-voice-transformation"},\n    {"name": "Blog Hub", "url": "https://www.sarvam.ai/blogs"},\n    {"name": "Conversational Agents", "url": "https://www.sarvam.ai/products/conversational-agents"},\n    {"name": "Studio", "url": "https://www.sarvam.ai/products/studio"},\n    {"name": "Akshar", "url": "https://www.sarvam.ai/products/akshar"},\n    {"name": "Text-to-Speech API", "url": "https://www.sarvam.ai/apis/text-to-speech"},\n    {"name": "Speech-to-Text API", "url": "https://www.sarvam.ai/apis/spee

In [17]:
# explore the context length and number of tokens
import tiktoken

print(f"Number of characters in context: {len(company_context)}")
print(f"Number of words in context: {len(company_context.split())}")

enc = tiktoken.encoding_for_model("gpt-5-nano")

print(f"Number of tokens: {len(enc.encode(company_context))}")


Number of characters in context: 56928
Number of words in context: 6998
Number of tokens: 12118


## Generate the brochure by invoking the LLM

In [18]:
brochure_gen_system_prompt = """
You are an expert in creating engaging & energetic short brochures for companies (for prospective clients, investors, recruits, etc.) based on information in relavant pages of the company's website.

Create the brochure and respond in Markdown format.

Include emojis to keep content engaging.
"""

In [20]:
def construct_brochure_gen_user_prompt(website_name, primary_url):
  extracted_relavant_pages = filter_relavant_links_for_brochure(website_name, primary_url)
  return f"""
You are looking at a company called: {website_name}

Below is the content of various relavant links of the company.

Based on this knowledge, create a short brochure and output in Markdown format. Do not include additional chat messages.

Company Details:
{generate_website_content_context(website_name, extracted_relavant_pages['links'])}
  """

In [21]:
def create_brochure(website_name, primary_url):
  response = llm.chat.completions.create(
    model="gpt-5-nano",
    messages=[
      {"role": "system", "content": brochure_gen_system_prompt},
      {"role": "user", "content": construct_brochure_gen_user_prompt(website_name, primary_url)}
    ]
  )
  response_text = response.choices[0].message.content
  # display the brochure in markdown
  display(Markdown(response_text))

In [22]:
create_brochure("Sarvam.AI", "https://www.sarvam.ai/")

Response:
ChatCompletion(id='chatcmpl-DTif79bseXzDCU1NVFxsVsUCOUPLI', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "links": [\n    {"name": "About Us", "url": "https://www.sarvam.ai/about-us"},\n    {"name": "Careers", "url": "https://www.sarvam.ai/careers"},\n    {"name": "Tata Capital AI Voice Transformation (Case Study)", "url": "https://www.sarvam.ai/stories/tata-capital-ai-voice-transformation"},\n    {"name": "Conversations Agents", "url": "https://www.sarvam.ai/products/conversational-agents"},\n    {"name": "Sarvam Studio", "url": "https://www.sarvam.ai/products/studio"},\n    {"name": "Akshar", "url": "https://www.sarvam.ai/products/akshar"},\n    {"name": "Text to Speech API", "url": "https://www.sarvam.ai/apis/text-to-speech"},\n    {"name": "Speech to Text API", "url": "https://www.sarvam.ai/apis/speech-to-text"},\n    {"name": "Document Digitisation API", "url": "https://www.sarvam.ai/apis/document-digitisation"}

# Sarvam AI — AI for India 🇮🇳🚀

Welcome to Sarvam AI, where India’s language, culture, and needs power a world-class, sovereign AI platform. We’re building for India — for everyday use, for the community, and for enterprises that demand scale with trust.

---

## Why Sarvam AI? Why now
- Built in India, governed in India — a full-stack AI platform designed for India’s languages, contexts, and workflows.
- 11+ Indian languages in translation & dubbing with high-fidelity voices and 23 languages across our document digitisation stack.
- Sovereign AI for developers, researchers, startups, and governments — with enterprise-grade security and data privacy.
- One platform, multiple surfaces — voice, chat, documents, and media, all integrated with your existing tools.

---

## Our Products

### Sarvam Samvaad — Conversational Agents 🤖💬
- End-to-end, full-stack platform for deploying AI agents across Voice, WhatsApp, Web, and more.
- Real-time, human-like conversations with sub-second latency; multi-language with Indian language support.
- Deep enterprise integration: pull/push data to CRMs, core banking, payment systems; multi-agent orchestration.
- Use cases: inbound support, loan journeys, bill payments, appointment bookings, and more.
- Security and governance built-in: enterprise-ready, scalable, and auditable.

### Sarvam Studio — Multilingual Content Studio 🎬🗣️
- AI-powered dubbing and translation for Indian languages (11+ languages supported).
- Voice cloning for consistent speaker identity across languages; precise audio-visual sync.
- Production-ready output with automated quality checks; layout-aware translation for video and documents.
- Use cases: education content, government/public communications, publishers, corporate L&D.
- Studio is currently in beta; join the waitlist or try the dashboard.

### Sarvam Akshar — Document Digitisation Platform 📄🔍
- Intelligent digitisation: reads complex layouts, dense tables, and Indic scripts; preserves reading order.
- Output formats: HTML, JSON, Markdown with layout and table structure preserved.
- Language coverage: 23 languages (22 Indian languages + English); supports complex tables and multi-column pages.
- API-first, agent-assisted corrections; visual grounding to review extracted data.
- Ideal for government records, publishing, finance/legal docs, and archives.

### Text to Speech API — Bulbul v3 🎤🗣️
- 11 Indian languages with expressive, natural voices and authentic pronunciation.
- 35+ voices; control of pace, tone, and delivery to match your brand.
- Low-latency streaming and REST options; multiple audio formats (MP3, WAV, etc.).
- Use cases: voice agents, content narrations, announcements, edtech, media, and more.

### Speech to Text API — Saaras v3 🗣️➡️📝
- 22 Indian languages with automatic language detection and code-mixing support.
- Real-time streaming or batch transcription; speaker diarization available.
- Robust in challenging acoustics: background noise, cross-talk, and variable quality.
- Use cases: live agents, post-call analytics, subtitling, accessibility.

### Document Digitisation API — 23-language Powerhouse 🧭🧩
- Async, batch-friendly workflow to convert scans, PDFs, and archives into structured data.
- Handles complex layouts, tables, and native Indic scripts; outputs HTML/Markdown/JSON.
- 23 languages; input formats include PDF, PNG, JPG, ZIP; secure, production-grade APIs.

---

## Real-World Impact: Tata Capital Case Study 💼💬
- Objective: multilingual voice engagement at scale across India.
- Outcome: high-volume, personalized conversations in English and 10 Indian languages; improved reach and engagement with cost efficiency.
- Key benefits: scalable voice AI, consistent, auditable interactions, and higher agent productivity by routing nuanced cases to humans.
- Quote: “Voice-driven AI is reshaping customer engagement… break access barriers, deepening engagement in a cost-effective manner.” — Shallu Kaushik, Chief Digital Officer, Tata Capital.
- Product at work: Samvaad as the enterprise-grade voice engagement layer.

---

## How It Works — Built to Scale, Right Now
- Full-stack platform: deploy, manage, and govern AI across voice, chat, and document workflows from a single ecosystem.
- Native India focus: models trained on Indian languages, dialects, and cultural nuances.
- Secure by design: SOC 2 compliant, end-to-end encryption, and data privacy built-in.
- Flexible deployment: Cloud (Sarvam Cloud, Private Cloud, On-Prem) to fit regulated environments.

---

## Pricing at a Glance 💸
- Pay-per-use credits with ₹1,000 free introductory credits; credits never expire and roll over.
- Plans: Starter (pay-as-you-go), Pro (₹10,000), and Business (₹50,000) with increasing rate limits and support levels.
- Universal credits for all APIs; no monthly commitments; higher rate limits available via custom plans.
- API-specific pricing highlights:
  - Vision (Document Digitisation): ₹1.50 per page
  - Text to Speech: ₹30 per 10k characters
  - Speech to Text: ₹30 per hour (± diarization options)
  - Language translation and transliteration: competitive per-10k-character rates
- Free tier included; no credit card required to start; easy top-ups from the dashboard.

---

## For Builders, Brands, and Public Services
- Ideal for enterprises needing multilingual, scalable AI in India.
- Government, education, finance, media, and consumer services benefit from sovereign AI capabilities.
- Ready-to-integrate with OpenAI-compatible APIs and official SDKs (Python, Node.js).

---

## Get Started
- Studio access: join the waitlist to experience Studio in beta.
- API access: sign up and receive instant API keys; start digitising or building agents in minutes.
- Resources: interactive docs, code samples, and quick-start guides for zero-to-first-extraction in under 5 minutes.
- Connect with Sales to tailor a plan for production workloads or enterprise requirements.

---

## Join Sarvam AI — Build the Future of India’s AI Today 🇮🇳✨
- Website: Sarvam AI
- Languages: 23 languages across our stack; 11+ languages supported in translation/dubbing; robust speech & vision models for Indian contexts.
- Vision: AI for India that’s accessible, affordable, and sovereign — empowering developers, enterprises, and communities to create with full agency.

If you’re ready to unlock India’s multilingual AI potential, let’s talk. Get started with Sarvam AI and build the future of India’s AI together. 🚀🤝

In [23]:
create_brochure("Langchain", "https://www.langchain.com/")

Response:
ChatCompletion(id='chatcmpl-DTig2BK019zhy6Vd24kPlmo38ffpo', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "links": [\n    {"name": "LangChain Home", "url": "https://www.langchain.com/"},\n    {"name": "About LangChain", "url": "https://www.langchain.com/about"},\n    {"name": "Startups", "url": "https://www.langchain.com/startups"},\n    {"name": "Careers", "url": "https://www.langchain.com/careers"},\n    {"name": "LangChain Partner Network", "url": "https://www.langchain.com/langchain-partner-network"},\n    {"name": "LangSmith Platform", "url": "https://www.langchain.com/langsmith-platform"},\n    {"name": "LangSmith Observability", "url": "https://www.langchain.com/langsmith/observability"},\n    {"name": "LangSmith Evaluation", "url": "https://www.langchain.com/langsmith/evaluation"},\n    {"name": "LangSmith Deployment", "url": "https://www.langchain.com/langsmith/deployment"},\n    {"name": "LangSmith Fleet", 

# LangChain: The Agent Engineering Platform 🚀🤖

Observability. Evaluation. Deployment. Fleet. Deep Agents. LangChain brings production-ready tools to build, debug, and ship reliable AI agents faster.

---

## Why LangChain? Quick snapshot

- Observe, evaluate, and deploy agents with ease across the entire lifecycle. 🧭
- Framework-agnostic, production-grade runtimes, and open-source options you can trust. 🔧
- Enterprise-ready: data residency options, SOC 2 Type 2, HIPAA, GDPR, and self-hosted capabilities. 🛡️
- Trusted by the ecosystem: 100M+ open source downloads, 6K+ LangSmith customers, and 35% of the Fortune 500. 🏢💼

---

## The LangSmith Platform: Built for real-world AI teams

- **Observability**: See exactly what your agents do with native tracing, OpenTelemetry, and multi-framework support. Debug with step-by-step timelines and AI-assisted insights. 📈
- **Evaluation**: Test and improve agent behavior with human feedback, LLM-as-judge evaluators, and online/offline evals. Calibrate prompts and compare versions side-by-side. 🧪
- **Deployment**: Ship in minutes with durable runtimes, exactly-once execution, and human-in-the-loop when needed. Centralized registry, versioning, and enterprise-grade controls. 🚢
- **Fleet**: No-code automation for entire teams. Describe tasks in plain language and have autonomous agents handle repeated work across tools and apps. 🧰

Extras:
- Native support for 30+ API endpoints, MCP, A2A, and Agent Protocols.
- Production-grade infrastructure that scales with your workloads. 🔄
- Memory, streaming, and orchestration built in for rich agent experiences. 🧠💬

Learn more across: LangSmith Platform, Observability, Evaluation, Deployment, Fleet, and Deep Agents. 🌐

---

## Open Source Frameworks & LangChain Ecosystem

- Deep Agents: autonomous, long-running agents with planning, memory, and subagents. 🧭
- LangChain: quick-start agents with any model provider for rapid prototyping. ⚡
- LangGraph: low-level orchestration for production-grade, deterministic behavior. 🛡️
- LangChain Academy, Documentation, Community, and a thriving ecosystem. 📚🌍

“

LangChain is streets ahead with LangGraph and LangSmith — a complete out-of-the-box solution to iterate quickly, debug immediately, and scale effortlessly. 
” — industry teammates

---

## Real-World Impact: Proven outcomes

- Klarna’s AI assistant reduced case resolution time by 80%. ⏱️
- Monday.com achieved faster feedback loops for evaluations. ⚡
- Podium cut engineering escalations by 90%. 🧰
- C.H. Robinson automated 5,500 orders/day, saving 600+ hours daily. 🗂️

Plus many more stories across Finance, eCommerce, Logistics, and Enterprise tech. 💬

---

## Start Building Today

- Start with LangSmith: observe, evaluate, and deploy agents in one click. 🧰
- Deploy on your terms: SaaS with data residency in US/EU, hybrid, or self-hosted deployments. 🗂️
- Fleet enables non-technical teams to build and run agents from templates (no code required). 🚦

Ready to ship reliable agents faster? Get a demo and see LangSmith in action. 👉 Get a demo

---

## Startups: Accelerate with Startup-friendly Pricing & Credits

- LangSmith for Startups: discounted pricing, credits, and exclusive programmer support. 💡
- Tiers to fit early-stage to Scale: Build and Scale programs with generous trace allowances and dedicated benefits. 🚀

How it works (high level):
1) Apply for Build or Scale tier. 2) Create a LangSmith account with your startup domain. 3) Activate and start building with LangSmith. 📝

Learn more and apply to the Startup program. 💼

---

## Pricing at a Glance

- Developer: Free to start. Core LangSmith features with base traces. 👶
- Plus: $39/seat/month + pay-as-you-go traces. Unlimited Fleet, team collaboration, and production-ready capabilities. 🧑‍💼
- Enterprise: Custom hosting, SSO/RBAC, SLAs, and tailored support. Custom fleet packages for large teams. 🛡️

Note: LangSmith traces, deployment runs, and Fleet usage scale with plan. Model usage billed separately by your provider. 🧾

Start with a plan that fits your team and scale as you grow. 🚀

---

## Data, Security & Compliance

- Data residency options: cloud (US/EU) or self-hosted on Kubernetes for fully private deployments. 🗺️
- Data ownership: LangSmith does not train on your data. You own your traces and prompts. 🔒
- Compliance-ready: HIPAA, SOC 2 Type 2, GDPR. Enterprise-ready controls and audit logs. 📜

---

## Learn, Connect, Build

- LangChain Academy: self-paced courses to level up your skills. 🎓
- Community & Events: meetups, forums, Slack, and conferences (e.g., Interrupt 2026 on May 13–14). 🗓️
- Blog, Customer Stories, Guides, and Documentation to stay on the cutting edge. 🧠

Join thousands of builders who are shaping the future of agent-driven AI. 🌟

---

## Ready to transform your AI agents?

- Get a demo: see LangSmith in action and tailor a solution for your team. 🎯
- Start building: explore LangSmith Platform, Observability, Evaluation, Deployment, Fleet, and Deep Agents today. 🧭

Contact the LangChain sales team to learn more and schedule a tailored walkthrough. 📞

Thank you for exploring LangChain — your partner in building reliable, scalable AI agents. 🧡